# Productos RLA — EDA y estandarización (Problema 1)
**Parte A — EDA (secciones 1–10):** calidad y estructura de `Lista_Productos.xlsx`.
**Parte B — Estandarización (secciones 11–19):** país y tipo de sitio, codificación, nomenclatura,
categorización, código maestro, duplicados, búsqueda y catálogo consolidado exportado a CSV.

**Cómo usar:** abrir en VS Code con las extensiones *Python* y *Jupyter*, elegir un kernel con
`pandas`, `openpyxl`, `matplotlib` e `ipykernel` y pulsar **Run All**.
Si falta algo: `python -m pip install pandas openpyxl matplotlib ipykernel`.

**Con un archivo nuevo:** cambiar `EXCEL_PATH` en la primera celda de código y ejecutar todo.
Los resultados quedan en la carpeta `RLA_estandarizacion`, junto al notebook.

No modifica el Excel. Una fila del Excel es una observación **producto × sitio**:
un código repetido en varios sitios no es un duplicado.

In [ ]:
from pathlib import Path
import os
import hashlib
import re
import unicodedata
from decimal import Decimal

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)
plt.rcParams.update({'figure.figsize': (11, 4.5), 'axes.spines.top': False, 'axes.spines.right': False})

# Ruta al Excel de R2: cambiarla aquí o definir la variable de entorno RLA_EXCEL.
EXCEL_PATH = Path(os.environ.get('RLA_EXCEL', Path.home() / 'Downloads' / 'Lista_Productos.xlsx'))
SHEET = 'Lista de productos'
hallazgos = []  # se completa a lo largo del notebook y se resume al final


def hallazgo(tema, detalle, n=None):
    hallazgos.append({'tema': tema, 'detalle': detalle, 'n': n})

# Parte A — EDA

## 1. Carga y procedencia
Todo se lee como texto para no perder ceros iniciales en los códigos (`01415641`) ni el
formato original de los números (`5.000,00`). Las conversiones se hacen después, en columnas nuevas.

In [ ]:
file_hash = hashlib.sha256(EXCEL_PATH.read_bytes()).hexdigest()
raw = pd.read_excel(EXCEL_PATH, sheet_name=SHEET, dtype=str, keep_default_na=False)
raw.index = pd.RangeIndex(2, len(raw) + 2, name='fila_excel')  # fila 1 = encabezados
print(f'Archivo : {EXCEL_PATH}\nSHA-256 : {file_hash}')
print(f'Filas   : {len(raw):,}\nColumnas: {raw.shape[1]}')
assert raw.columns.is_unique, 'Hay encabezados repetidos'
display(raw.head())

In [ ]:
# Copia de trabajo: se recortan espacios y los vacíos pasan a NA. `raw` queda intacto.
df = raw.apply(lambda s: s.str.strip()).replace('', pd.NA)

## 2. Perfil de columnas
Completitud, cardinalidad y columnas constantes o casi vacías (candidatas a no migrar).

In [ ]:
perfil = pd.DataFrame({
    'no_vacios': df.notna().sum(),
    'pct_vacio': (df.isna().mean() * 100).round(1),
    'distintos': df.nunique(dropna=True),
    'valor_mas_comun': df.mode(dropna=True).iloc[0] if len(df) else None,
})
perfil['frec_mas_comun'] = [df[c].value_counts().iloc[0] if df[c].notna().any() else 0 for c in df.columns]
perfil['pct_mas_comun'] = (perfil['frec_mas_comun'] / perfil['no_vacios'].replace(0, np.nan) * 100).round(1)
display(perfil.sort_values('pct_vacio', ascending=False))

vacias = perfil.index[perfil['no_vacios'] == 0].tolist()
constantes = perfil.index[perfil['distintos'] == 1].tolist()
casi_vacias = perfil.index[(perfil['pct_vacio'] >= 95) & (perfil['no_vacios'] > 0)].tolist()
print('Columnas totalmente vacías :', vacias)
print('Columnas con un solo valor :', constantes)
print('Columnas ≥95 % vacías      :', casi_vacias)
hallazgo('Columnas', f'{len(vacias)} vacías, {len(constantes)} constantes, {len(casi_vacias)} ≥95 % vacías', len(vacias) + len(constantes) + len(casi_vacias))

In [ ]:
ax = perfil['pct_vacio'].sort_values().plot.barh(figsize=(9, 10), color='#4C78A8')
ax.set_xlabel('% de filas vacías'); ax.set_title('Completitud por columna')
plt.tight_layout(); plt.show()

## 3. Conversión numérica
Los importes vienen como texto con formato chileno (`5.000,00`). Se convierten a número en
columnas `*_num`. Los valores que no calzan con el patrón se listan, no se descartan en silencio.

In [ ]:
NUMERICAS = ['Stock', 'Cost', 'Replacement Cost', 'CostoTotal', 'RETAILPRICE', 'LOWRETAILPRICE',
             'MSRP', 'MAXIMUMQTY', 'MINIMUMQTY', 'REORDERQTY']
NUMERICAS = [c for c in NUMERICAS if c in df.columns]
PATRON_CL = re.compile(r'^-?\d{1,3}(\.\d{3})*(,\d+)?$|^-?\d+(,\d+)?$')
PATRON_PUNTO = re.compile(r'^-?\d+\.\d+$')  # p. ej. 8180.0 exportado por Excel


def a_numero(valor):
    if pd.isna(valor):
        return np.nan
    s = str(valor)
    if PATRON_CL.match(s):
        return float(Decimal(s.replace('.', '').replace(',', '.')))
    if PATRON_PUNTO.match(s):
        return float(s)
    return 'ERROR'


no_convertibles = {}
for col in NUMERICAS:
    convertido = df[col].map(a_numero)
    errores = convertido == 'ERROR'
    if errores.any():
        no_convertibles[col] = df.loc[errores, col].value_counts().head(10)
    df[col + '_num'] = pd.to_numeric(convertido.where(~errores), errors='coerce')

if no_convertibles:
    for col, vals in no_convertibles.items():
        print(f'\n{col}: valores no convertibles'); display(vals)
    hallazgo('Números', 'Hay valores con formato no reconocido: ' + ', '.join(no_convertibles), sum(v.sum() for v in no_convertibles.values()))
else:
    print('Todos los valores numéricos se convirtieron correctamente.')

NUM = [c + '_num' for c in NUMERICAS]
display(df[NUM].describe(percentiles=[.25, .5, .75, .95, .99]).T.round(2))

In [ ]:
resumen_num = pd.DataFrame({
    'vacios': df[NUM].isna().sum(),
    'ceros': (df[NUM] == 0).sum(),
    'negativos': (df[NUM] < 0).sum(),
    'positivos': (df[NUM] > 0).sum(),
})
resumen_num['pct_ceros'] = (resumen_num['ceros'] / len(df) * 100).round(1)
display(resumen_num)
neg_stock = int((df['Stock_num'] < 0).sum())
hallazgo('Stock', 'Filas con stock negativo (¿sobre-asignación, préstamo, error?)', neg_stock)
hallazgo('Costos', f"{resumen_num.loc['Cost_num', 'pct_ceros']} % de filas con Cost = 0", int(resumen_num.loc['Cost_num', 'ceros']))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['Stock_num', 'Cost_num', 'Replacement Cost_num']):
    serie = df.loc[df[col] > 0, col]
    ax.hist(np.log10(serie), bins=50, color='#4C78A8')
    ax.set_title(f'{col} > 0 (n={len(serie):,})'); ax.set_xlabel('log10(valor)')
plt.tight_layout(); plt.show()

### 3.1 Stock negativo
¿Dónde se concentra? Sirve para decidir si se bloquea, se advierte o se acepta al publicar.

In [ ]:
neg = df[df['Stock_num'] < 0]
display(neg.groupby('SITENAME')['Stock_num'].agg(filas='count', suma='sum').sort_values('filas', ascending=False).head(15))
display(neg[['Product ID', 'Description', 'SITEID', 'Stock']].sort_values('Product ID').head(20))

### 3.2 ¿CostoTotal = Stock × Cost?
Antes de usar CostoTotal hay que saber qué representa. Se compara con una tolerancia de 1 peso.

In [ ]:
calc = df['Stock_num'] * df['Cost_num']
comparable = df['CostoTotal_num'].notna() & calc.notna()
# Tolerancia relativa: Cost se muestra con 2 decimales pero el sistema guarda más; con stock alto
# eso produce diferencias de algunos pesos que no son errores.
dif = (df['CostoTotal_num'] - calc).abs()
coincide = (dif <= 1) | (dif <= 0.001 * df['CostoTotal_num'].abs())
print(f'Filas comparables: {comparable.sum():,}')
print(f'Coinciden con Stock×Cost: {(coincide & comparable).sum():,} ({(coincide & comparable).sum() / comparable.sum():.1%})')
difieren = df[comparable & ~coincide]
display(difieren[['Product ID', 'SITEID', 'Stock', 'Cost', 'CostoTotal']].head(15))
hallazgo('CostoTotal', 'Filas donde CostoTotal ≠ Stock × Cost (tolerancia 0,1 %): es un dato derivable', len(difieren))

## 4. Claves e identidad
- ¿Hay filas sin código o sin sitio?
- ¿Se repite la combinación código–sitio?
- ¿Un mismo código tiene descripciones, tipo, marca o modelo distintos entre sitios?

In [ ]:
sin_codigo = df['Product ID'].isna().sum()
sin_sitio = df['SITEID'].isna()
print(f'Códigos distintos : {df["Product ID"].nunique():,}')
print(f'Sitios distintos  : {df["SITEID"].nunique():,}')
print(f'Filas sin código  : {sin_codigo}')
print(f'Filas sin sitio   : {sin_sitio.sum()}')
display(df.loc[sin_sitio, ['Product ID', 'Description', 'Type', 'Stock']])
hallazgo('Claves', 'Filas sin SITEID (bloquean publicación)', int(sin_sitio.sum()))

dup_clave = df[df.duplicated(['Product ID', 'SITEID'], keep=False) & df['SITEID'].notna()]
print(f'Filas con código–sitio repetido: {len(dup_clave)}')
hallazgo('Claves', 'Filas con código–sitio repetido', len(dup_clave))

ceros = df['Product ID'].fillna('').str.fullmatch(r'0+')
print(f'Filas con código formado solo por ceros: {ceros.sum()}')
display(df.loc[ceros, ['Product ID', 'Description', 'SITEID']].head(10))
hallazgo('Claves', 'Filas con código "0"', int(ceros.sum()))

In [ ]:
ATRIBUTOS_IDENTIDAD = ['Description', 'Type', 'ITEMCATEGORY', 'Package', 'MANUFACTURER', 'MODEL',
                       'CANRENT', 'CANSELL', 'CANSUBRENT']
inconsistencia = df.groupby('Product ID')[ATRIBUTOS_IDENTIDAD].nunique(dropna=True)
conteo_incons = (inconsistencia > 1).sum().sort_values(ascending=False)
display(conteo_incons.to_frame('códigos con >1 valor'))
for col, n in conteo_incons.items():
    if n:
        hallazgo('Identidad', f'Códigos con {col} distinto entre sitios', int(n))

# Ejemplos para la columna más conflictiva
peor = conteo_incons.index[0]
if conteo_incons.iloc[0]:
    codigos = inconsistencia.index[inconsistencia[peor] > 1][:5]
    display(df[df['Product ID'].isin(codigos)][['Product ID', 'SITEID', peor]].drop_duplicates(['Product ID', peor]))

## 5. Dominios categóricos
Tipo, paquete, serialización y banderas operativas.

In [ ]:
CATEGORICAS = ['Type', 'Package', 'ITEMCATEGORY', 'CANRENT', 'CANSELL', 'CANSUBRENT',
               'AFFECTSAVAILABILITY', 'ISFREIGHT', 'ISMISCITEM']
CATEGORICAS = [c for c in CATEGORICAS if c in df.columns]
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for ax, col in zip(axes.flat, CATEGORICAS):
    df[col].fillna('(vacío)').value_counts().plot.bar(ax=ax, color='#72B7B2')
    ax.set_title(col); ax.tick_params(axis='x', rotation=30)
for ax in axes.flat[len(CATEGORICAS):]:
    ax.axis('off')
plt.tight_layout(); plt.show()

# Tabla cruzada útil para la taxonomía: tipo × paquete, contando códigos únicos
codigos = df.drop_duplicates('Product ID')
display(pd.crosstab(codigos['Type'].fillna('(vacío)'), codigos['Package'].fillna('(vacío)'), margins=True))

## 6. Sitios
Tamaño de cada sitio (códigos y stock) y sitios con nombres inconsistentes.

In [ ]:
sitios = df.groupby('SITEID').agg(
    nombre=('SITENAME', lambda s: ' | '.join(sorted(s.dropna().unique()))),
    filas=('Product ID', 'size'),
    codigos=('Product ID', 'nunique'),
    stock_total=('Stock_num', 'sum'),
    filas_stock_pos=('Stock_num', lambda s: (s > 0).sum()),
).sort_values('filas', ascending=False)
sitios['pct_con_stock'] = (sitios['filas_stock_pos'] / sitios['filas'] * 100).round(1)
display(sitios.head(20))

varios_nombres = df.groupby('SITEID')['SITENAME'].nunique()
print(f'SITEID con más de un SITENAME: {(varios_nombres > 1).sum()}')
sin_stock = sitios[sitios['filas_stock_pos'] == 0]
print(f'Sitios sin ninguna fila con stock > 0: {len(sin_stock)}')
hallazgo('Sitios', 'Sitios sin ninguna fila con stock positivo', len(sin_stock))
hallazgo('Sitios', 'SITEID con más de un nombre', int((varios_nombres > 1).sum()))

ax = sitios['filas'].head(25).iloc[::-1].plot.barh(figsize=(9, 8), color='#4C78A8')
ax.set_title('25 sitios con más filas'); ax.set_xlabel('filas')
plt.tight_layout(); plt.show()

In [ ]:
por_codigo = df.groupby('Product ID')['SITEID'].nunique()
print(por_codigo.describe().round(2))
ax = por_codigo.clip(upper=60).plot.hist(bins=60, color='#72B7B2')
ax.set_title('Sitios por código (recortado en 60)'); ax.set_xlabel('nº de sitios')
plt.tight_layout(); plt.show()

## 7. Fabricante y modelo
Faltantes, marcadores de ausencia y variantes de escritura de una misma marca.

In [ ]:
MARCADORES = {'NA', 'N/A', 'NULL', 'NONE', 'S/N', 'SN', 'S/M', 'SM', 'SIN MARCA', 'SIN DATO', '-', '--', '---', '----',
              '.', 'GENERICO', 'GENÉRICO', 'GENERICA', 'GENÉRICA', '0'}


def clave(texto):
    if pd.isna(texto):
        return pd.NA
    t = unicodedata.normalize('NFKD', str(texto).upper())
    t = ''.join(c for c in t if not unicodedata.combining(c))
    return re.sub(r'[^A-Z0-9]', '', t)


for col in ['MANUFACTURER', 'MODEL']:
    s = codigos[col]
    marcador = s.str.upper().isin(MARCADORES)
    print(f'{col}: vacíos {s.isna().sum():,} | marcadores {marcador.sum():,} | distintos {s.nunique():,} (códigos únicos: {len(codigos):,})')
    hallazgo('Identidad', f'Códigos sin {col} o con marcador de ausencia', int(s.isna().sum() + marcador.sum()))

display(codigos['MANUFACTURER'].value_counts().head(25).to_frame('códigos'))

In [ ]:
# Variantes: textos distintos que colapsan a la misma clave (sin tildes, espacios ni signos)
marcas = codigos['MANUFACTURER'].dropna().to_frame()
marcas['clave'] = marcas['MANUFACTURER'].map(clave)
variantes = (marcas.groupby('clave')['MANUFACTURER']
             .agg(variantes=lambda s: sorted(s.unique()), n_variantes='nunique', codigos='size')
             .query('n_variantes > 1').sort_values('codigos', ascending=False))
print(f'Marcas con variantes de escritura: {len(variantes)}')
display(variantes.head(25))
hallazgo('Marcas', 'Grupos de marcas con variantes de escritura (candidatos a alias)', len(variantes))

## 8. Descripciones
Calidad del texto y códigos distintos con la misma descripción (candidatos a revisar como duplicados).

In [ ]:
desc = codigos['Description'].fillna('')
largo = desc.str.len()
print(largo.describe().round(1))
raros = desc.str.contains(r'[^\w\s\.,;:\-/\(\)\"\'#%&+°º´`!?¿¡*=<>@\[\]]', regex=True)
print(f'Descripciones con caracteres fuera de lo común: {raros.sum():,}')
display(codigos.loc[raros, ['Product ID', 'Description']].head(10))
tilde_grave = desc.str.contains('[òàèìù]')
print(f'Descripciones con tilde grave (probable error de codificación, p. ej. "Iluminaciòn"): {tilde_grave.sum():,}')
hallazgo('Texto', 'Descripciones con tilde grave (ò, à…) probablemente mal codificadas', int(tilde_grave.sum()))
eliminar = desc.str.contains(r'\belimin|\bno\s*usar\b|\bobsolet|\bbaja\b|ficha mala', case=False)  # \b evita contar "bajada"
print(f'Descripciones que sugieren dar de baja ("eliminar", "no usar", "obsoleto"...): {eliminar.sum():,}')
display(codigos.loc[eliminar, ['Product ID', 'Description']].head(15))
hallazgo('Texto', 'Descripciones que piden eliminar / no usar el producto', int(eliminar.sum()))

In [ ]:
codigos_desc = codigos.assign(desc_clave=codigos['Description'].map(clave))
desc_invalida = codigos_desc['desc_clave'].fillna('').str.len() < 3  # ".", "-" o vacía: dato faltante, no duplicado
print(f'Códigos con descripción inválida: {desc_invalida.sum()}')
hallazgo('Texto', 'Códigos con descripción inválida (".", vacía)', int(desc_invalida.sum()))
misma_desc = codigos_desc[codigos_desc.duplicated('desc_clave', keep=False) & ~desc_invalida]
grupos = misma_desc.groupby('desc_clave')
print(f'Grupos de descripción idéntica (normalizada) con más de un código: {grupos.ngroups:,}')
print(f'Códigos involucrados: {len(misma_desc):,}')
display(misma_desc.sort_values('desc_clave')[['Product ID', 'Description', 'MANUFACTURER', 'MODEL', 'Type']].head(30))
hallazgo('Duplicados', 'Grupos de códigos con descripción idéntica normalizada', grupos.ngroups)

mismo_mm = codigos_desc.dropna(subset=['MANUFACTURER', 'MODEL'])
mismo_mm = mismo_mm[~mismo_mm['MANUFACTURER'].str.upper().isin(MARCADORES) & ~mismo_mm['MODEL'].str.upper().isin(MARCADORES)]
mismo_mm = mismo_mm.assign(mm=mismo_mm['MANUFACTURER'].map(clave) + '|' + mismo_mm['MODEL'].map(clave))
dup_mm = mismo_mm[mismo_mm.duplicated('mm', keep=False)]
print(f'Grupos marca+modelo con más de un código: {dup_mm["mm"].nunique():,} ({len(dup_mm):,} códigos)')
display(dup_mm.sort_values('mm')[['Product ID', 'Description', 'MANUFACTURER', 'MODEL']].head(20))
hallazgo('Duplicados', 'Grupos marca+modelo compartidos por varios códigos', dup_mm['mm'].nunique())

## 9. Agrupaciones y taxonomía original
Cardinalidad de cada campo de agrupación y si es constante por código
(si varía entre sitios, no puede usarse directamente como categoría del producto).

In [ ]:
AGRUPACIONES = ['Availability Group', 'Report Group', 'REVENUEGROUP', 'EXCHANGEGROUP', 'INVENTORYGROUP',
                'DEPARTMENT', 'DISCOUNTGROUP', 'TAXGROUP', 'COGSGROUP', 'Price Group']
AGRUPACIONES = [c for c in AGRUPACIONES if c in df.columns]
variacion = df.groupby('Product ID')[AGRUPACIONES].nunique(dropna=True)
tax = pd.DataFrame({
    'distintos': df[AGRUPACIONES].nunique(),
    'pct_vacio': (df[AGRUPACIONES].isna().mean() * 100).round(1),
    'códigos_que_varían': (variacion > 1).sum(),
})
display(tax.sort_values('pct_vacio'))

for col in ['Report Group', 'REVENUEGROUP', 'EXCHANGEGROUP']:
    if col in codigos:
        print(f'\n{col} — 15 valores más frecuentes (códigos únicos)')
        display(codigos[col].fillna('(vacío)').value_counts().head(15).to_frame('códigos'))

In [ ]:
# ¿Report Group y REVENUEGROUP cuentan la misma historia? Útil para elegir la fuente de la taxonomía.
if {'Report Group', 'REVENUEGROUP'} <= set(codigos.columns):
    cruce = pd.crosstab(codigos['REVENUEGROUP'].fillna('(vacío)'), codigos['Report Group'].fillna('(vacío)'))
    top = cruce.sum().sort_values(ascending=False).head(12).index
    display(cruce[top].loc[cruce[top].sum(axis=1) > 0])

## 10. Registros técnicos
Códigos de sistema como DEFAULTITEM / SYSTEMDEFAULT: ¿cuántos hay y qué contienen?

In [ ]:
tecnicos = df[df['Product ID'].str.upper().str.contains('DEFAULT', na=False)]
display(tecnicos[['Product ID', 'Description', 'Type', 'SITEID', 'Stock']])
hallazgo('Técnicos', 'Filas de códigos de sistema (DEFAULT*)', len(tecnicos))

# Parte B — Estandarización (Problema 1)
El Problema 1 pide una estructura de productos común a todos los países que estandarice
**codificación, nomenclatura y categorización**, facilite crear y buscar productos y se mantenga
ordenada en el tiempo. Esta parte convierte los hallazgos del EDA en un proceso repetible:

**Excel nuevo → procesamiento → estandarización → consolidación → catálogo actualizado**

Todas las reglas están en diccionarios y listas editables (celdas marcadas **REGLAS**). Al llegar
un archivo nuevo basta con cambiar `EXCEL_PATH` y ejecutar todo: las reglas se aplican igual,
los códigos maestros ya asignados se conservan (registro persistente) y solo lo nuevo
o lo dudoso queda en la lista de pendientes para revisión humana.

Principio: **nada se fusiona ni se borra automáticamente**. Las reglas proponen y el revisor decide.

In [ ]:
SALIDA = Path.cwd() / 'RLA_estandarizacion'   # carpeta junto al notebook
SALIDA.mkdir(exist_ok=True)


def clave_texto(texto):
    """Mayúsculas, sin tildes y sin signos, pero conservando espacios: sirve para reglas y búsqueda."""
    if pd.isna(texto):
        return ''
    t = unicodedata.normalize('NFKD', str(texto).upper())
    t = ''.join(c for c in t if not unicodedata.combining(c))
    return re.sub(r'\s+', ' ', re.sub(r'[^A-Z0-9]+', ' ', t)).strip()

print('Resultados en:', SALIDA)

## 11. Sitios: país y tipo de ubicación
El Excel no trae el país. Tampoco distingue una bodega de un hotel cliente, de un sitio de
descarte ("BOTAR CHILE") o de un sitio contable ("GHOST 2024", "DIFERENCIAS INV").
Sin esto no se puede consolidar por país ni calcular el stock realmente disponible.

Prioridad para el país: **1)** asignación manual, **2)** palabra clave geográfica en el nombre,
**3)** evidencia de los códigos: si al menos el 30 % de las filas del sitio usan códigos con prefijo
de país y el 70 % de esos prefijos es de un mismo país. Lo que no cumple nada queda como `REVISAR`.

In [ ]:
# ---------- REGLAS (editables) ----------
PAIS_MANUAL = {
    # 'SITEID': 'CL',   ← completar aquí los sitios que queden en REVISAR
}
REGLAS_PAIS = [  # (regex sobre SITEID + SITENAME sin tildes, país). Gana la primera coincidencia.
    (r'PANAM', 'PA'),
    (r'MEXICO', 'MX'),
    (r'MIAMI', 'US'),
    (r'PERU|LIMA|MIRAFLO|SAN ISIDRO|\bAQP\b|INKA|INCA OLD|CASA ANDINA|COSTA DEL SOL', 'PE'),
    (r'COLOM|BOGOTA|^H ?C |H\.C\.|CARTAGEN|MEDELLIN|EFCOL|\bCOL ?22\b|FONTANA|SUITE JONES|PARQUE 93', 'CO'),
    (r'CHILE|SANTIAGO|STGO|CONCEPCION|\bCNC\b|ANTOFAGAST|VINA DEL MAR|TEMUCO|VALDIVIA|PTO VARAS|PTA ARENAS|'
     r'PTO MONTT|RENACA|PUCON|COYHAIQUE|CHILLAN|VITACURA|LAS CONDES|LA DEHESA|PROVIDENCIA|RINCONADA|'
     r'O HIGGINS|PARQUE ARAUCO|MANQUEHUE|OFICINA CENTRAL', 'CL'),
]
REGLAS_TIPO = [  # (tipo, regex). Gana la primera coincidencia; el resto se considera sede de operación.
    ('Descarte / baja', r'BOTAR|REMATE|FUERA DE USO|EQUIPO MALO|REBAJAR'),
    ('Venta', r'\bVENTA\b'),
    ('Servicio técnico', r'SERVICIO TECNICO|SSTT|REPAIR|MERMA'),
    ('Ajuste / no ubicado', r'GHOST|INV 2023|INVENTARIO 20|DIFERENCIAS|AJUSTE|RASTREAR|FUERA COL|FUERA PERU|'
                            r'NO USAR|TRANSMET|TRANSITO|\bOLD\b|PROYECTOS 20'),
    ('Administrativo', r'ADMINISTRACI|CORPORATIVO|OFICINA CENTRAL|RLA MEXICO'),
    ('Bodega / CD', r'^CD |\bCD\b|BODEGA|REUTILIZAR|\bSAV\b'),
]
TIPOS_DISPONIBLES = {'Bodega / CD', 'Sede de operación'}  # stock que cuenta como disponible
# -----------------------------------------

PREFIJO_PAIS = r'^(CO|PE|CL|PA|MX)[\s_\-]|(?<=[_\d])(CO|CL|PE|PA|MX)$'
df['pais_codigo'] = df['Product ID'].str.extract(PREFIJO_PAIS).bfill(axis=1).iloc[:, 0]

sit = df.groupby('SITEID').agg(
    SITENAME=('SITENAME', 'first'), filas=('Product ID', 'size'),
    stock=('Stock_num', 'sum'), filas_stock_pos=('Stock_num', lambda s: int((s > 0).sum())),
    con_prefijo=('pais_codigo', lambda s: s.notna().mean()),
    pais_prefijo=('pais_codigo', lambda s: s.value_counts(normalize=True).idxmax() if s.notna().any() else None),
    dominancia=('pais_codigo', lambda s: s.value_counts(normalize=True).max() if s.notna().any() else 0),
)


def pais_sitio(siteid, fila):
    if siteid in PAIS_MANUAL:
        return PAIS_MANUAL[siteid], 'manual'
    texto = clave_texto(f'{siteid} {fila.SITENAME}')
    texto_raw = f'{siteid} {fila.SITENAME}'.upper()
    for patron, pais in REGLAS_PAIS:
        if re.search(patron, texto) or re.search(patron, texto_raw):
            return pais, 'palabra clave'
    if fila.con_prefijo >= 0.30 and fila.dominancia >= 0.70:
        return fila.pais_prefijo, 'prefijo de códigos'
    return 'REVISAR', 'sin evidencia'


def tipo_sitio(siteid, fila):
    texto = clave_texto(f'{siteid} {fila.SITENAME}')
    for tipo, patron in REGLAS_TIPO:
        if re.search(patron, texto):
            return tipo
    return 'Sede de operación'


sit[['pais', 'fuente_pais']] = [pais_sitio(i, f) for i, f in sit.iterrows()]
sit['tipo'] = [tipo_sitio(i, f) for i, f in sit.iterrows()]
df = df.join(sit[['pais', 'tipo']].rename(columns={'pais': 'pais_sitio', 'tipo': 'tipo_sitio'}), on='SITEID')

display(pd.crosstab(sit['pais'], sit['fuente_pais'], margins=True))
display(sit.pivot_table(index='tipo', columns='pais', values='stock', aggfunc='sum', fill_value=0, margins=True).round(0))

revisar = sit[sit['pais'] == 'REVISAR'].sort_values('filas', ascending=False)
print(f'Sitios sin país asignado: {len(revisar)} de {len(sit)} — completar PAIS_MANUAL')
display(revisar[['SITENAME', 'filas', 'stock', 'tipo']])
stock_no_disp = df.loc[~df['tipo_sitio'].isin(TIPOS_DISPONIBLES) & (df['Stock_num'] > 0), 'Stock_num'].sum()
print(f'Stock positivo en sitios NO disponibles (descarte, ajuste, técnico, venta, admin.): {stock_no_disp:,.0f} '
      f'({stock_no_disp / df.loc[df.Stock_num > 0, "Stock_num"].sum():.1%} del total)')
hallazgo('Sitios', 'Sitios sin país determinable (requieren mapeo manual)', len(revisar))
hallazgo('Sitios', 'Sitios que no son operativos (descarte, ajuste, técnico, venta, administrativo)',
         int((~sit['tipo'].isin(TIPOS_DISPONIBLES)).sum()))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
sit.groupby('pais')['filas'].sum().sort_values().plot.barh(ax=axes[0], color='#4C78A8', title='Filas por país del sitio')
(df[df.Stock_num > 0].groupby('tipo_sitio')['Stock_num'].sum().sort_values()
   .plot.barh(ax=axes[1], color='#F58518', title='Stock positivo por tipo de sitio'))
plt.tight_layout(); plt.show()

## 12. Codificación actual: cada país codifica distinto
Se clasifica cada código según su patrón y se cruza con el país del sitio donde aparece.
Esto muestra que la codificación depende del país y que el mismo equipo puede existir con varios códigos.

In [ ]:
def patron_codigo(c):
    c = str(c)
    if re.fullmatch(r'\d{5}', c): return '5 dígitos (10063)'
    if re.fullmatch(r'0\d+', c): return 'numérico con cero inicial'
    if re.fullmatch(r'\d{6,}', c): return 'numérico largo (6+)'
    if re.fullmatch(r'\d+\.\d+', c): return 'numérico con sufijo (10193.1)'
    if re.match(r'(CO|PE|CL|PA|MX)[\s_\-]', c): return 'prefijo país (CO PRV07)'
    if re.search(r'(?<=[_\d])(CO|CL|PE|PA|MX)$', c): return 'sufijo país (PALED2.9_M2CO)'
    if re.fullmatch(r'[A-Z]+', c): return 'solo letras'
    return 'alfanumérico libre'


df['patron_codigo'] = df['Product ID'].map(patron_codigo)
cruce = pd.crosstab(df['patron_codigo'], df['pais_sitio'])
display(cruce)
display((cruce / cruce.sum() * 100).round(1).astype(str) + ' %')

# Mismo producto (descripción + marca + modelo) con códigos distintos en distintos países
cod = df.drop_duplicates('Product ID').copy()
cod['clave_producto'] = (cod['Description'].map(clave) + '|' + cod['MANUFACTURER'].map(clave).fillna('') + '|'
                         + cod['MODEL'].map(clave).fillna(''))
multi = cod[cod['Description'].map(clave).fillna('').str.len() >= 3].groupby('clave_producto').agg(
    codigos=('Product ID', 'nunique'), paises_codigo=('pais_codigo', lambda s: s.fillna('sin prefijo').nunique()),
    ejemplos=('Product ID', lambda s: ', '.join(s.head(4))))
multi = multi[multi['codigos'] > 1]
print(f'Productos idénticos (desc+marca+modelo) con más de un código: {len(multi):,}; '
      f'de ellos con esquemas de país distintos: {(multi.paises_codigo > 1).sum():,}')
display(multi.sort_values('codigos', ascending=False).head(15))
hallazgo('Codificación', 'Esquemas de codificación distintos en uso', df['patron_codigo'].nunique())
hallazgo('Codificación', 'Productos idénticos con más de un código', len(multi))

## 13. Nomenclatura: nombre estándar
Reglas de texto aplicadas a la descripción. Separan tres cosas que hoy están mezcladas en el nombre:
- **el producto** (lo que queda en el nombre estándar);
- **el país** (`(CL)`, `(CO)`): pasa a ser atributo del sitio, no del nombre;
- **el estado operativo** (`NO USAR`, `eliminar`, `ficha mala`): pasa a una marca de estado.

Además se unifican unidades (`mts` → `m`, `¨` → `in`, `Ansilumenes` → `ANSI lm`), géneros de conector
(`M-M` → `macho-macho`), mayúsculas y errores de tilde. Se agregan marca y modelo al final si no estaban.

In [ ]:
# ---------- REGLAS (editables) ----------
REEMPLAZOS_TEXTO = [  # (regex, reemplazo), en orden
    (r'ò', 'ó'), (r'à', 'á'), (r'è', 'é'), (r'ì', 'í'), (r'ù', 'ú'),
    (r'[–—]', '-'), (r'\\', '/'), (r'®|™', ''),
    (r'\s*\((?:CL|CO|PE|PA|MX)\)', ''),                                   # etiqueta de país
    (r'\(?\s*\bno\s*usar(?:\s*definitivo)?\s*\)?|\beliminar\b|\bficha mala\b', ' '),  # estado operativo
    (r'(\d)\s*(?:¨|\'\'|"+|”|pulgadas?\b|pulg\b\.?)', r'\1 in '),
    (r'(\d)\s*(?:mtrs?\.?|mts?\.?|metros?)(?=\W|$)', r'\1 m'),
    (r'(\d)\s*(?:cms?\.?|cent[ií]metros?)(?=\W|$)', r'\1 cm'),
    (r'\b(\d{1,2})\.(\d{3})\s*(?=ansi|l[uú]m)', r'\1\2 '),                # 6.000 lúmenes → 6000
    (r'\bansi\s*l[uú]menes\b|\bansil[uú]menes\b|\bl[uú]menes\s*ansi\b', 'ANSI lm'),
    (r'\bM\s*-\s*M\b', 'macho-macho'), (r'\bM\s*-\s*H\b', 'macho-hembra'), (r'\bH\s*-\s*H\b', 'hembra-hembra'),
    (r'\s*/\s*', ' / '), (r'\s*\|\s*', ' / '), (r'\(\s*\)', ''), (r'\s+', ' '), (r'^[\s\-/.,]+|[\s\-/.,]+$', ''),
]
SIGLAS = {'HDMI', 'VGA', 'LED', 'LCD', 'USB', 'SDI', 'XGA', 'WXGA', 'WUXGA', 'DMX', 'UPS', 'TV', 'BNC', 'XLR',
          'RCA', 'DVI', 'HD', 'ANSI', 'AWG', 'PC', 'CPU', 'IP', 'POE', 'DJ', 'UHF', 'VHF', 'RF', 'AV', 'DVD',
          'PAR', 'RGB', 'RGBW', 'UTP', 'CAT', 'DSP', 'EDID', 'MIDI', 'PTZ', 'NFC', 'GB', 'TB', 'W', 'VA', 'ML'}
TILDES = {  # palabra sin tilde → con tilde (se respeta la mayúscula inicial)
    'microfono': 'micrófono', 'microfonos': 'micrófonos', 'inalambrico': 'inalámbrico', 'alambrico': 'alámbrico',
    'dinamico': 'dinámico', 'camara': 'cámara', 'telon': 'telón', 'energia': 'energía', 'iluminacion': 'iluminación',
    'traduccion': 'traducción', 'interpretacion': 'interpretación', 'proyeccion': 'proyección', 'conexion': 'conexión',
    'extension': 'extensión', 'bateria': 'batería', 'audifono': 'audífono', 'audifonos': 'audífonos', 'tripode': 'trípode',
    'electrico': 'eléctrico', 'electrica': 'eléctrica', 'optica': 'óptica', 'senal': 'señal', 'modulo': 'módulo',
    'modulos': 'módulos', 'salon': 'salón', 'lamina': 'lámina', 'metalico': 'metálico', 'plastico': 'plástico',
    'tactil': 'táctil', 'portatil': 'portátil', 'computacion': 'computación', 'edicion': 'edición', 'grafica': 'gráfica',
    'tecnico': 'técnico', 'sonorizacion': 'sonorización', 'informatica': 'informática', 'musica': 'música', 'transmision': 'transmisión', 'votacion': 'votación', 'direccion': 'dirección',
}
ESTADO_OPERATIVO = r'\bno\s*usar|\belimin|\bficha mala\b|\bobsolet|\bprobar\b|item missing'
# -----------------------------------------


def casing(texto):
    """Si viene TODO EN MAYÚSCULAS, pasa a tipo oración respetando siglas y tokens con dígitos."""
    letras = [c for c in texto if c.isalpha()]
    if not letras or sum(c.isupper() for c in letras) / len(letras) < 0.7:
        return texto[:1].upper() + texto[1:]
    palabras = []
    for w in texto.split(' '):
        base = clave_texto(w)
        palabras.append(w if base in SIGLAS or re.search(r'\d', w) else w.lower())
    t = ' '.join(palabras)
    return t[:1].upper() + t[1:]


def palabra_std(w):
    base = clave_texto(w)
    if base in SIGLAS:
        return re.sub(r'[A-Za-z]+', lambda m: m.group().upper(), w)
    fijo = TILDES.get(w.lower())
    if fijo:
        return fijo.capitalize() if w[:1].isupper() else fijo
    return w


def nombre_normalizado(desc):
    if pd.isna(desc) or len(clave(desc) or '') < 3:
        return pd.NA
    t = str(desc)
    for patron, reemplazo in REEMPLAZOS_TEXTO:
        t = re.sub(patron, reemplazo, t, flags=re.IGNORECASE)
    t = ' '.join(palabra_std(w) for w in casing(t.strip()).split(' '))
    return (t[:1].upper() + t[1:]) or pd.NA


ejemplos = ['Microfono Alambrico Dinámico', 'Cable BNC M-M 30 metros', 'Proyector 6.000 Ansilumenes / Formato 4:3 / XGA LCD',
            'TV de 42¨', 'Panel LED 50X50 Pixel 2.9 D2V (CL)', 'NO USARProyector LCD 5000 ansilúmenes',
            'SISTEMA DE SONORIZACION PARA FIESTAS (sala de baile)', 'Cable USB 5 Mts']
display(pd.DataFrame({'original': ejemplos, 'estándar': [nombre_normalizado(e) for e in ejemplos]}))

### 13.1 Marcas y modelos
Las variantes que colapsan a la misma clave (`DA-LITE`, `DA LITE`, `DALITE`) se unifican
automáticamente con la escritura más frecuente. `ALIAS_MARCA` corrige lo que la clave no detecta
(errores de ortografía, nombres largos y cortos). `MARCA_VACIA` lista los rellenos que significan "sin marca".

In [ ]:
# ---------- REGLAS (editables) ----------
ALIAS_MARCA = {  # clave (sin espacios ni signos) → nombre canónico
    'BLACKMAGIC': 'BLACKMAGIC DESIGN', 'BLACKMAGICDESIGN': 'BLACKMAGIC DESIGN',
    'ALLENHEAT': 'ALLEN & HEATH', 'ALLENHEATH': 'ALLEN & HEATH', 'POLARLIGTH': 'POLAR LIGHT',
    'DALITE': 'DA-LITE', 'TPLINK': 'TP-LINK', 'DLINK': 'D-LINK', 'ELECTROVOICE': 'ELECTRO-VOICE',
    'TVONE': 'TV ONE', 'DSAN': 'DSAN', 'ANALOGWAY': 'ANALOG WAY',
}
MARCA_VACIA = {clave(m) for m in MARCADORES} | {'', 'SM', 'SINMARCA', 'GENERICO', 'GENERICA'}
# -----------------------------------------

marca_clave = cod['MANUFACTURER'].map(clave).fillna('')
mas_frecuente = cod.assign(k=marca_clave).dropna(subset=['MANUFACTURER']).groupby('k')['MANUFACTURER'].agg(lambda s: s.value_counts().index[0])
cod['marca_std'] = [pd.NA if k in MARCA_VACIA else ALIAS_MARCA.get(k, mas_frecuente.get(k, pd.NA)) for k in marca_clave]
modelo = cod['MODEL'].str.upper().str.replace(r'\s*-\s*', '-', regex=True).str.replace(r'\s+', ' ', regex=True)
cod['modelo_std'] = modelo.where(~modelo.map(clave).fillna('').isin(MARCA_VACIA))

antes, despues = cod['MANUFACTURER'].nunique(), cod['marca_std'].nunique()
print(f'Marcas distintas: {antes} → {despues} tras estandarizar ({antes - despues} variantes unificadas)')
hallazgo('Nomenclatura', 'Variantes de marca unificadas', antes - despues)

In [ ]:
def nombre_estandar(fila):
    base = nombre_normalizado(fila['Description'])
    if pd.isna(base):
        return pd.NA
    k = clave(base)
    extra = [x for x in (fila['marca_std'], fila['modelo_std']) if pd.notna(x) and clave(x) and clave(x) not in k]
    return ' '.join([base] + extra)


cod['nombre_std'] = cod.apply(nombre_estandar, axis=1)
cod['estado_operativo'] = np.where(cod['Description'].fillna('').str.contains(ESTADO_OPERATIVO, case=False, regex=True),
                                   'Revisar: marcado no usar / eliminar', 'Activo')
cod.loc[cod['nombre_std'].isna(), 'estado_operativo'] = 'Revisar: descripción inválida'
display(cod[['Product ID', 'Description', 'MANUFACTURER', 'MODEL', 'nombre_std']].sample(15, random_state=1))
n_orig = cod['Description'].nunique()
n_norm = cod['Description'].map(nombre_normalizado).map(clave).nunique()
print(f'Descripciones distintas: {n_orig:,} tal como vienen → {n_norm:,} tras normalizar '
      f'({n_orig - n_norm:,} eran la misma descripción escrita de otra forma)')
hallazgo('Nomenclatura', 'Descripciones que solo diferían en la escritura', n_orig - n_norm)

## 14. Categorización: una familia repartida en muchas categorías
Primero la evidencia del problema: se detecta la familia por la descripción y se cuenta
en cuántas etiquetas originales distintas quedó repartida cada una.

In [ ]:
# ---------- REGLAS (editables) ----------
# Taxonomía: categoría → familia → (código de categoría, código de familia). Tomada del repositorio CASO-RLA.
TAXONOMIA = {
    'Micrófonos': ('Audio', 'AUD', 'MIC'), 'Altavoces': ('Audio', 'AUD', 'ALT'),
    'Consolas de audio': ('Audio', 'AUD', 'CSL'), 'Amplificadores': ('Audio', 'AUD', 'AMP'),
    'Procesamiento de audio': ('Audio', 'AUD', 'PRA'), 'Accesorios de audio': ('Audio', 'AUD', 'ACA'),
    'Proyectores': ('Video', 'VID', 'PRY'), 'Monitores y televisores': ('Video', 'VID', 'MON'),
    'Pantallas LED': ('Video', 'VID', 'LED'), 'Pantallas de proyección': ('Video', 'VID', 'TEL'),
    'Cámaras': ('Video', 'VID', 'CAM'), 'Procesamiento y distribución de video': ('Video', 'VID', 'PRV'),
    'Reproducción de video': ('Video', 'VID', 'REP'), 'Accesorios de video': ('Video', 'VID', 'ACV'),
    'Computadores': ('Informática y redes', 'INF', 'COM'), 'Redes': ('Informática y redes', 'INF', 'RED'),
    'Impresoras': ('Informática y redes', 'INF', 'IMP'), 'Periféricos y almacenamiento': ('Informática y redes', 'INF', 'PER'),
    'Luminarias': ('Iluminación', 'ILU', 'LUM'), 'Control de iluminación': ('Iluminación', 'ILU', 'CTL'),
    'Accesorios de iluminación': ('Iluminación', 'ILU', 'ACI'),
    'Distribución eléctrica': ('Energía', 'ENE', 'DIS'), 'Respaldo y generación': ('Energía', 'ENE', 'RES'),
    'Interpretación simultánea': ('Interpretación y comunicaciones', 'ITC', 'SIM'),
    'Debate y votación': ('Interpretación y comunicaciones', 'ITC', 'DEB'),
    'Conferencia e intercomunicación': ('Interpretación y comunicaciones', 'ITC', 'CNF'),
    'Cables y adaptadores': ('Conectividad', 'CNX', 'CAB'),
    'Estructuras y montaje': ('Montaje y soporte', 'MNT', 'EST'),
    'Transporte y protección de equipos': ('Montaje y soporte', 'MNT', 'CAS'),
    'Mobiliario y oficina': ('Mobiliario y oficina', 'MOB', 'MOB'),
    'Consumibles y repuestos': ('Consumibles y repuestos', 'CSM', 'CSM'),
    'Servicios técnicos': ('Servicios', 'SRV', 'TEC'), 'Personal': ('Servicios', 'SRV', 'PSN'),
    'Transporte y viajes': ('Servicios', 'SRV', 'TRV'), 'Licencias': ('Servicios', 'SRV', 'LIC'),
    'Gastos y cargos': ('Cargos comerciales', 'CAR', 'GAS'),
    'Paquetes y soluciones': ('Paquetes y soluciones', 'PAQ', 'SOL'),
}
PATRONES_FAMILIA = {  # sobre el inicio del nombre estándar (sin tildes, mayúsculas)
    'Debate y votación': r'^(MICROFONO.*DEBATE|UNIDAD (DE )?(DEBATE|VOTACION)|SISTEMA DE (DEBATE|VOTACION)|DICENTIS)\b',
    'Micrófonos': r'^MICROFONO\b',
    'Altavoces': r'^(ALTAVOZ|ALTAVOCES|PARLANTE|SUBWOOFER|SUB BAJO|SOUNDBAR|LINE ARRAY)\b',
    'Consolas de audio': r'^(CONSOLA (DE )?AUDIO|MEZCLADORA? (DE )?AUDIO|POWER MIXER|MIXER)\b',
    'Amplificadores': r'^AMPLIFICADOR\b',
    'Procesamiento de audio': r'^(PROCESADOR (DE )?AUDIO|CAJA DIRECTA|COMPRESOR|ECUALIZADOR|MATRIZ (DE )?AUDIO)\b',
    'Accesorios de audio': r'^(RECEPTOR.*(MICROFONO|MIC\b)|ATRIL (DE )?MICROFONO|SOPORTE (DE )?(MICROFONO|PARLANTE)|ANTENA|AUDIFONO)',
    'Proyectores': r'^(PROYECTOR|VIDEOPROYECTOR)\b',
    'Monitores y televisores': r'^(MONITOR|TELEVISOR|TV|PANTALLA TACTIL|PANTALLA \d|PLASMA|TOTEM)\b',
    'Pantallas LED': r'^(PANTALLA LED|PANEL LED|MODULOS? (DE )?LED|PANTALLA PIXEL)\b',
    'Pantallas de proyección': r'^(TELON|ECRAN|PANTALLA (DE )?PROYECCION|PANTALLA INFLABLE)\b',
    'Cámaras': r'^(CAMARA|VIDEOCAMARA|CCTV)\b',
    'Procesamiento y distribución de video': r'^(SWITCHER|ESCALADOR|MATRIZ (DE )?VIDEO|DISTRIBUIDOR (DE )?(VIDEO|VGA|HDMI|SDI)|PROCESADOR (DE )?VIDEO|EXTENSOR|CAPTURADORA|(CONVERSOR|CONVERTIDOR))\b',
    'Reproducción de video': r'^(REPRODUCTOR|SISTEMA WATCHOUT)\b',
    'Accesorios de video': r'^(LENTE|SOPORTE (DE )?(PROYECTOR|PLASMA|LCD|MONITOR|TV))\b',  # adaptadores → Conectividad
    'Computadores': r'^(NOTEBOOK|LAPTOP|COMPUTADORA?|SERVIDOR|CPU|MAC ?BOOK|IMAC)\b',
    'Redes': r'^(SWITCH (DE )?RED|SWITCH \d|ROUTER|ACCESS POINT|PUNTO DE ACCESO)\b',
    'Impresoras': r'^(IMPRESORA|MULTIFUNCIONAL)\b',
    'Periféricos y almacenamiento': r'^(MOUSE|TECLADO|MEMORIA|PENDRIVE|DISCO DURO|LECTOR|TABLET|IPAD|PUNTERO)\b',
    'Luminarias': r'^(FOCO|LUMINARIA|CABEZA MOVIL|OPTIPAR|PAR LED|PAR \d|ROBOTICO|ROBOTIZADO|LASER|WASH|BEAM|REFLECTOR|FRESNEL)\b',
    'Control de iluminación': r'^(CONSOLA (DE )?ILUMINACION|DIMMER|SPLITTER|CONTROLADOR DMX)\b',
    'Accesorios de iluminación': r'^SOPORTE (DE )?ILUMINACION\b',
    'Distribución eléctrica': r'^(TABLERO|DISTRIBUIDOR (ELECTRICO|DE CORRIENTE)|CAJA DE DISTRIBUCION|ZAPATILLA)\b',
    'Respaldo y generación': r'^(UPS|GENERADOR)\b',
    'Interpretación simultánea': r'^(RECEPTOR (DE )?(TRADUCCION|INFRARROJO|SENAL)|PUPITRE|CABINA|RADIADOR|TRANSMISOR (DE )?(TRADUCCION|INFRARROJO)|AUDIFONO PARA RECEPTOR|IDIOMA)',
    'Conferencia e intercomunicación': r'^(INTERCOMUNICADOR|RADIO|TELEFONO|SISTEMA (DE )?VIDEOCONFERENCIA|VIDEOCONFERENCIA)\b',
    'Cables y adaptadores': r'^(CABLE|CHICOTE|EXTENSION|CORDON|COPLA|ADAPTADOR|CONECTOR)\b',
    'Estructuras y montaje': r'^(TRUSS|ESTRUCTURA|TORRE|BASE (DE )?PISO|TRIPODE|PEDESTAL|RIGGING|RACK)\b',
    'Transporte y protección de equipos': r'^(CASE|FLIGHT CASE|BOLSO|MALETA|ESTUCHE|GABINETE)\b',
    'Mobiliario y oficina': r'^(MESA|SILLA|PAPELOGRAFO|ROTAFOLIO|PIZARRA|PODIO|ATRIL(?! DE MICROFONO)|BIOMBO|HOJAS)\b',
    'Consumibles y repuestos': r'^(TINTA|TONER|REPUESTO|PILA|BATERIA|CINTA|HUINCHA|AMARRA|AMPOLLETA|LAMPARA)\b',
    'Servicios técnicos': r'^(SERVICIO|CONFIGURACION|EDICION|HORA DE EDICION|MAPPING|MAPING|REGISTRO|GRABACION|STREAMING|ELECTROMONTAJE|DISENO|LANDING)\b',
    'Personal': r'^(OPERADOR|TECNICO|INTERPRETE|PERSONAL|COORDINADOR|DJ|PRODUCTOR|ASISTENTE)\b',
    'Transporte y viajes': r'^(TRANSPORTE|VIAJE|FLETE|TRASLADO)\b',
    'Licencias': r'^LICENCIA\b',
    'Gastos y cargos': r'^(COMISION|ALOJAMIENTO|ALIMENTACION|IMPREVISTOS|VIATICO|SUBARRIENDO)\b',
    'Paquetes y soluciones': r'(?!)',  # nunca por patrón principal: solo por respaldo (PATRONES_RESPALDO)
}
ALIAS_ETIQUETA = {  # etiqueta original (Report Group / EXCHANGEGROUP) → familia
    'Micrófonos': ['MICROFONOS', 'AUDIO MICROPHONE', 'MICROFONOS INALAMBRICOS', 'MICROFONOS ALAMBRICOS', 'MICROFONOS DE PODIUM', 'MICROFONO PODIUM', 'MIC PODIUM', 'MICROFONO HEADSET', 'MICROFONO DE CAMARA'],
    'Altavoces': ['ALTAVOCES', 'SUB BAJO ACTIVO', 'SOUNDBAR', 'SISTEMA ARRAY', 'LINE ARRY'],
    'Consolas de audio': ['CONSOLAS AUDIO', 'CONSOLAS DE AUDIO', 'POWER MIXER'],
    'Amplificadores': ['AMPLIFICADOREWS', 'SISTEMA DE AMPLIFICACION', 'AMPLIFICADORES'],
    'Procesamiento de audio': ['PROCESADORES DE AUDIO', 'CAJA DIRECTA', 'MATRIZ DE AUDIO', 'CONTROLADOR MIDI'],
    'Accesorios de audio': ['ACCESORIOS AUDIO', 'ACCESORIOS DE AUDIO', 'PERIFERICOS AUDIO', 'RECEPTOR MIC', 'ANTENA AUDIO', 'DISTRIBUIDOR DE ANTENAS', 'ATRIL DE MICROFONO', 'SOPORTE PARLANTE', 'AUDIFONOS'],
    'Proyectores': ['PROYECTORES', 'PROJECTOR', 'PROYECTORES LASER'],
    'Monitores y televisores': ['MONITORES Y TV', 'TELEVISORES MONITORES', 'PANTALLA TACTIL', 'PANTALLAS TACTILES', 'MONITORES'],
    'Pantallas LED': ['PANTALLA LED'],
    'Pantallas de proyección': ['TELONES', 'TELON ELECTRICO', 'TELON MECANO', 'TELON MANUAL', 'PANTALLA INFLABLE'],
    'Cámaras': ['CAMARA DE VIDEO', 'FOTOGRAFIA'],
    'Procesamiento y distribución de video': ['DISTRIBUIDOR DE VIDEO', 'EXTENSORES DE VIDEO', 'PROCESADOR DE VIDEO', 'SWITH DE VIDEO', 'MATRIZ DE VIDEO', 'ESCALADORES', 'ESCALER', 'SWITCHER/DISTRIBUIDOR'],
    'Reproducción de video': ['REPRODUCTORES VIDEO', 'SISTEMA WATCHOUT', 'DIGITAL SIGNAGE'],
    'Accesorios de video': ['ACCESORIOS DE VIDEO', 'ADAPTADORES DE VIDEO', 'SOPORTE PLASMA', 'SOPORTE LCD', 'SOPORTE PROYECTOR', 'SOPORTE WALL 4'],
    'Computadores': ['NOTEBOOK', 'LAPTOP', 'CPU', 'COMPUTADORES', 'SERVIDORES'],
    'Redes': ['REDES', 'SWITCH DE RED', 'ROUTERS', 'NETWORKING'],
    'Impresoras': ['IMPRESORA', 'IMPRESORAS'],
    'Periféricos y almacenamiento': ['PERIFERICOS PC', 'MOUSE INALAMBRICO', 'MEMRORIAS', 'DISPOSITIVOS MOVILES', 'LECTORES CODIGOS', 'ACCESORIOS INFORMATICOS'],
    'Luminarias': ['FOCOS LED', 'FOCO PAR 64', 'ROBOTIZADO', 'LASER'],
    'Control de iluminación': ['CONSOLA DE ILUMINACION', 'SPLITTER ILUMINACION', 'POWER DIMMER', 'DIMMER'],
    'Accesorios de iluminación': ['PERIFERICO ILUMINACION', 'SOPORTE ILUMINACION'],
    'Distribución eléctrica': ['TABLERO ELECTRICO', 'PERIFERICOS ELECTRICIDAD'],
    'Respaldo y generación': ['UPS', 'GENERADOR'],
    'Interpretación simultánea': ['SISTEMA DE INTERPRETACION SIMULTANEA', 'TRADUCCION SIMULTANEA', 'AUDIFONO TRADUCCION', 'RECEPTOR DE TRADUCCION', 'PUPITRE', 'CABINAS'],
    'Debate y votación': ['MIC DEBATE', 'SISTEMA DE DEBATE', 'SISTEMA DEBATE', 'SISTEMA DE VOTACION'],
    'Conferencia e intercomunicación': ['VIDEOCONFERENCIA', 'VIDEO CONFERENCIA', 'TELEFONO CONFERENCIA', 'INTERCOMUNICADORES', 'RADIOS'],
    'Cables y adaptadores': ['CABLES', 'CABLE', 'CABLES DE AUDIO', 'CABLES VIDEO', 'CABLES DE ENERGIA', 'CABLES DE RED', 'CABLES DE DATOS', 'CABLES TRADUCCION', 'CABLES ILUMINACION', 'CABLE S'],
    'Estructuras y montaje': ['ACCESORIOS MONTAJE', 'RIGGING PANTALLA LED', 'ESTRUCTURA', 'BASE PISO'],
    'Transporte y protección de equipos': ['CASES', 'BOLSOS'],
    'Mobiliario y oficina': ['MOBILIARIO', 'ARTICULOS DE OFICINA', 'PAPELOGRAFO'],
    'Consumibles y repuestos': ['INSUMOS', 'REPUESTOS', 'TINTA IMPRESORA', 'BATERIAS'],
    'Servicios técnicos': ['SERVICIO TECNICO', 'CONFIGURACION REDES', 'EDICION', 'REGISTRO AUDIO', 'MAPING'],
    'Personal': ['LABOR', 'PERSONAL', 'OPERADOR', 'INTERPRETES', 'DJ'],
    'Transporte y viajes': ['TRANSPORTE', 'VIAJES', 'VIAJE SUR'],
    'Licencias': ['LICENCIA'],
    'Gastos y cargos': ['IMPREVISTOS', 'COMISION AGENCIA', 'ALOJAMIENTO', 'ALIMENTACION'],
    'Paquetes y soluciones': [],
}
# Respaldo: se aplica solo si ninguna familia específica coincide. Cubre soluciones armadas
# ("Sistema de amplificación…", "Salón…", "Kit…") que agrupan varios equipos.
PATRONES_RESPALDO = r'^(SISTEMA|SALON|SALA|KIT|PAQUETE|PACK|PLAN|SET|PROYECCION|AMPLIFICACION|SONIDO|AV|AUDIO Y VIDEO|ILUMINACION|AUDIO|VIDEO|STUDIO|ESTUDIO)\b'
# Etiquetas originales genéricas: si contradicen un nombre específico, gana el nombre.
FAMILIAS_GENERICAS = {'Accesorios de video', 'Accesorios de audio', 'Periféricos y almacenamiento',
                      'Accesorios de iluminación', 'Estructuras y montaje', 'Consumibles y repuestos'}
# -----------------------------------------
assert set(PATRONES_FAMILIA) == set(TAXONOMIA) == set(ALIAS_ETIQUETA), 'Taxonomía, patrones y alias deben tener las mismas familias'
ETIQUETA_A_FAMILIA = {clave_texto(a): fam for fam, alias in ALIAS_ETIQUETA.items() for a in alias}


def familia_por_nombre(nombre):
    k = clave_texto(nombre)
    hits = [f for f, p in PATRONES_FAMILIA.items() if re.search(p, k)]
    if 'Debate y votación' in hits:  # más específica que "Micrófonos"
        hits = ['Debate y votación']
    if not hits and re.search(PATRONES_RESPALDO, k):
        return 'Paquetes y soluciones'
    return hits[0] if len(hits) == 1 else (pd.NA if not hits else 'CONFLICTO:' + '/'.join(hits))


def familia_por_etiqueta(fila):
    fams = {ETIQUETA_A_FAMILIA.get(clave_texto(fila[c])) for c in ['Report Group', 'EXCHANGEGROUP'] if pd.notna(fila[c])}
    fams.discard(None)
    return fams.pop() if len(fams) == 1 else (pd.NA if not fams else 'CONFLICTO:' + '/'.join(sorted(fams)))


cod['fam_nombre'] = cod['nombre_std'].map(familia_por_nombre)
cod['fam_etiqueta'] = cod.apply(familia_por_etiqueta, axis=1)

# Evidencia de fragmentación: familias detectadas por nombre vs etiquetas originales distintas
ok = cod[cod['fam_nombre'].notna() & ~cod['fam_nombre'].astype(str).str.startswith('CONFLICTO')]
frag = ok.groupby('fam_nombre').agg(
    codigos=('Product ID', 'size'),
    REVENUEGROUP=('REVENUEGROUP', 'nunique'), DEPARTMENT=('DEPARTMENT', 'nunique'),
    Report_Group=('Report Group', 'nunique'), EXCHANGEGROUP=('EXCHANGEGROUP', 'nunique'),
    ejemplos_REVENUEGROUP=('REVENUEGROUP', lambda s: ', '.join(s.value_counts().index[:5].astype(str))),
).sort_values('codigos', ascending=False)
display(frag)
print('Ejemplo: los cables están repartidos en', frag.loc['Cables y adaptadores', 'REVENUEGROUP'], 'valores de REVENUEGROUP y',
      frag.loc['Cables y adaptadores', 'DEPARTMENT'], 'departamentos.')

### 14.1 Clasificación propuesta
| Estado | Regla |
|---|---|
| **Alta confianza** | nombre y etiqueta original indican la misma familia |
| **Por nombre** / **Por etiqueta** | solo una de las dos fuentes indica familia, sin contradicción |
| **Por nombre (etiqueta genérica)** | la etiqueta es un cajón genérico ("Accesorios de video") y el nombre es específico |
| **Por tipo** | `Type` = LABOR → Personal; `Package` = PACKAGE → Paquetes y soluciones (sin otra evidencia) |
| **Revisar: conflicto** | las fuentes indican familias distintas |
| **Revisar: sin regla** | ninguna regla aplica: se clasifica a mano o se crea una regla nueva |

In [ ]:
def clasificar(fila):
    n, e = fila['fam_nombre'], fila['fam_etiqueta']
    n_ok = pd.notna(n) and not str(n).startswith('CONFLICTO')
    e_ok = pd.notna(e) and not str(e).startswith('CONFLICTO')
    if pd.notna(n) and not n_ok:
        return pd.NA, 'Revisar: conflicto ' + str(n).replace('CONFLICTO:', '')
    if n_ok and e_ok:
        if n == e:
            return n, 'Alta confianza'
        if e in FAMILIAS_GENERICAS and n not in FAMILIAS_GENERICAS:
            return n, 'Por nombre (etiqueta genérica)'
        if n == 'Paquetes y soluciones':
            return e, 'Por etiqueta'
        return pd.NA, f'Revisar: conflicto ({n} vs {e})'
    if fila['Package'] == 'PACKAGE' and not n_ok and not e_ok:
        return 'Paquetes y soluciones', 'Por tipo'
    if n_ok:
        return n, 'Por nombre'
    if e_ok:
        return e, 'Por etiqueta'
    if pd.isna(n) and pd.isna(e):
        tipo = {'LABOR': 'Personal'}.get(fila['Type'])  # MISCCHARGE se revisa: hay equipos mal tipificados
        return (tipo, 'Por tipo') if tipo else (pd.NA, 'Revisar: sin regla')
    return pd.NA, 'Revisar: conflicto ' + str(n if pd.notna(n) else e).replace('CONFLICTO:', '')


cod[['familia', 'estado_clasif']] = cod.apply(clasificar, axis=1, result_type='expand')
cod['categoria'] = cod['familia'].map(lambda f: TAXONOMIA[f][0] if pd.notna(f) else pd.NA)
cod.loc[cod['estado_operativo'] == 'Revisar: descripción inválida', ['familia', 'categoria']] = pd.NA
cod.loc[cod['estado_operativo'] == 'Revisar: descripción inválida', 'estado_clasif'] = 'Revisar: descripción inválida'

def motivo_corto(m):
    m = str(m)
    return 'Revisar: conflicto' if m.startswith('Revisar: conflicto') else m.split(' (')[0].split(' DUP-')[0]


resumen_clasif = cod['estado_clasif'].map(motivo_corto).value_counts()
display(resumen_clasif.to_frame('códigos').assign(pct=lambda t: (t['códigos'] / len(cod) * 100).round(1)))
display(cod.groupby(['categoria', 'familia']).size().to_frame('códigos'))
cobertura = cod['familia'].notna().mean()
print(f'Cobertura de clasificación automática: {cobertura:.1%} de {len(cod):,} códigos')
hallazgo('Categorización', f'Códigos clasificados automáticamente ({cobertura:.0%})', int(cod['familia'].notna().sum()))
hallazgo('Categorización', 'Códigos que requieren clasificación manual', int(cod['familia'].isna().sum()))

ax = resumen_clasif.sort_values().plot.barh(color='#54A24B', figsize=(9, 4))
ax.set_title('Resultado de la clasificación'); ax.set_xlabel('códigos')
plt.tight_layout(); plt.show()

## 15. Codificación estándar y registro persistente
Formato propuesto, igual para todos los países: **`CAT-FAM-NNNNN`** (ej. `AUD-MIC-00012`).
- El país **no** va en el código: vive en el sitio. Un mismo equipo tiene un solo código en todos los países.
- El código se asigna solo cuando el producto tiene familia; los pendientes lo reciben al clasificarse.
- El registro `registro_codigos.csv` hace que la asignación sea **estable**: al procesar un Excel nuevo,
  los códigos ya asignados se mantienen y solo los productos nuevos reciben número. Si un producto
  cambia de familia conserva su código (no se re-codifica), para no romper cotizaciones ni historial.
- El código de origen de cada país queda como equivalencia (`codigo_origen → codigo_maestro`).

In [ ]:
REGISTRO = SALIDA / 'registro_codigos.csv'
if REGISTRO.exists():
    registro = pd.read_csv(REGISTRO, dtype=str)
else:
    registro = pd.DataFrame(columns=['codigo_origen', 'codigo_maestro', 'familia_asignacion', 'fecha_asignacion', 'archivo_sha256'])
print(f'Registro previo: {len(registro):,} códigos')

ya = set(registro['codigo_origen'])
# Solo productos activos y clasificados reciben código; los marcados "no usar" esperan decisión.
nuevos = cod[cod['familia'].notna() & (cod['estado_operativo'] == 'Activo') & ~cod['Product ID'].isin(ya)].sort_values('Product ID')
correlativo = (registro['codigo_maestro'].str.extract(r'^(\w{3}-\w{3})-(\d+)$').dropna()
               .astype({1: int}).groupby(0)[1].max().to_dict())
filas_nuevas = []
for _, f in nuevos.iterrows():
    _, cat, fam = TAXONOMIA[f['familia']]
    pref = f'{cat}-{fam}'
    correlativo[pref] = correlativo.get(pref, 0) + 1
    filas_nuevas.append({'codigo_origen': f['Product ID'], 'codigo_maestro': f'{pref}-{correlativo[pref]:05d}',
                         'familia_asignacion': f['familia'], 'fecha_asignacion': pd.Timestamp.now().isoformat(timespec='seconds'),
                         'archivo_sha256': file_hash})
registro = pd.concat([registro, pd.DataFrame(filas_nuevas)], ignore_index=True)
assert registro['codigo_maestro'].is_unique and registro['codigo_origen'].is_unique
registro.to_csv(REGISTRO, index=False, encoding='utf-8-sig')
print(f'Códigos nuevos asignados en esta ejecución: {len(filas_nuevas):,} · total en registro: {len(registro):,}')

cod = cod.drop(columns=['codigo_maestro'], errors='ignore').merge(
    registro[['codigo_origen', 'codigo_maestro']], left_on='Product ID', right_on='codigo_origen', how='left').drop(columns='codigo_origen')
display(cod.dropna(subset=['codigo_maestro'])[['Product ID', 'codigo_maestro', 'nombre_std', 'categoria', 'familia']].head(10))

## 16. Candidatos a duplicado (después de estandarizar)
Con nombre, marca y modelo ya estandarizados, los códigos que coinciden en los tres se agrupan como
**candidatos**. Ejemplo típico: el mismo panel LED con un código por país. No se fusionan solos:
se revisan y, si se aprueba, comparten código maestro.

In [ ]:
valida = cod['nombre_std'].notna() & (cod['estado_operativo'] == 'Activo')
cod['clave_dup'] = np.where(valida, cod['nombre_std'].map(clave) + '|' + cod['marca_std'].map(clave).fillna('')
                            + '|' + cod['modelo_std'].map(clave).fillna(''), None)
tam = cod.groupby('clave_dup')['Product ID'].transform('size')
es_dup = valida & (tam > 1)
ids = {k: f'DUP-{i:04d}' for i, k in enumerate(sorted(cod.loc[es_dup, 'clave_dup'].unique()), 1)}
cod['grupo_duplicado'] = cod['clave_dup'].map(ids)
print(f'Grupos candidatos a duplicado: {len(ids):,} ({es_dup.sum():,} códigos)')
entre_paises = cod[es_dup].groupby('grupo_duplicado')['pais_codigo'].apply(lambda s: s.fillna('-').nunique() > 1).sum()
print(f'Grupos que mezclan códigos de distintos países: {entre_paises}')
display(cod[es_dup].sort_values('grupo_duplicado')[['grupo_duplicado', 'Product ID', 'nombre_std', 'familia']].head(20))
hallazgo('Duplicados', 'Grupos candidatos a duplicado tras estandarizar', len(ids))

## 17. Buscar antes de crear
Para evitar fichas duplicadas, antes de crear un producto se busca en el catálogo estandarizado.
La búsqueda ignora tildes, mayúsculas, signos y orden de las palabras.

In [ ]:
def buscar(texto, n=10):
    palabras = clave_texto(texto).split()
    base = cod['nombre_std'].map(clave_texto) + ' ' + cod['Product ID'].map(clave_texto)
    mascara = np.logical_and.reduce([base.str.contains(rf'\b{re.escape(p)}', regex=True) for p in palabras])
    return cod.loc[mascara, ['Product ID', 'codigo_maestro', 'nombre_std', 'familia', 'estado_operativo']].head(n)


display(buscar('microfono inalambrico shure'))
display(buscar('cable hdmi 10 m'))

## 18. Catálogo consolidado y archivos de salida
- `catalogo_estandarizado.csv`: un registro por código de origen, con código maestro, nombre, marca,
  modelo, categoría, familia, estado y stock por tipo de sitio.
- `pendientes_revision.csv`: lo que necesita decisión humana (sin familia, conflicto, no usar,
  descripción inválida, candidato a duplicado).
- `sitios_estandarizados.csv`: país y tipo de cada sitio (base para completar `PAIS_MANUAL`).
- `registro_codigos.csv`: equivalencias permanentes código de origen → código maestro.

In [ ]:
stock_disp = (df[df['tipo_sitio'].isin(TIPOS_DISPONIBLES)].groupby('Product ID')['Stock_num']
              .apply(lambda s: s.clip(lower=0).sum()))
paises = df[df['Stock_num'] > 0].groupby('Product ID')['pais_sitio'].agg(lambda s: ', '.join(sorted(s.unique())))
catalogo = cod.assign(
    stock_disponible=cod['Product ID'].map(stock_disp).fillna(0),
    paises_con_stock=cod['Product ID'].map(paises),
)[['Product ID', 'codigo_maestro', 'nombre_std', 'Description', 'marca_std', 'modelo_std', 'categoria', 'familia',
   'estado_clasif', 'estado_operativo', 'grupo_duplicado', 'Type', 'Package', 'ITEMCATEGORY', 'pais_codigo',
   'stock_disponible', 'paises_con_stock']].rename(columns={'Product ID': 'codigo_origen', 'Description': 'descripcion_original'})

pendientes = catalogo[catalogo['familia'].isna() | (catalogo['estado_operativo'] != 'Activo') | catalogo['grupo_duplicado'].notna()].copy()
pendientes['motivo'] = np.select(
    [pendientes['estado_operativo'] != 'Activo', pendientes['familia'].isna(), pendientes['grupo_duplicado'].notna()],
    [pendientes['estado_operativo'], pendientes['estado_clasif'], 'Candidato a duplicado ' + pendientes['grupo_duplicado'].fillna('')])
# Prioridad: primero lo que tiene stock disponible (impacta la operación)
pendientes = pendientes.sort_values(['stock_disponible', 'codigo_origen'], ascending=[False, True])

catalogo.to_csv(SALIDA / 'catalogo_estandarizado.csv', index=False, encoding='utf-8-sig')
pendientes.to_csv(SALIDA / 'pendientes_revision.csv', index=False, encoding='utf-8-sig')
sit.reset_index().to_csv(SALIDA / 'sitios_estandarizados.csv', index=False, encoding='utf-8-sig')
print(f'Catálogo: {len(catalogo):,} códigos · con código maestro: {catalogo["codigo_maestro"].notna().sum():,}')
print(f'Pendientes de revisión: {len(pendientes):,}')
display(pendientes['motivo'].map(motivo_corto).value_counts().to_frame('códigos'))
display(catalogo.head(10))

## 19. Resumen para la toma de decisiones

In [ ]:
resumen = pd.DataFrame(hallazgos)
resumen = resumen[resumen['n'].fillna(1) > 0].reset_index(drop=True)
display(resumen)

display(Markdown(f'''
**Datos generales**
- {len(df):,} filas producto × sitio · {df["Product ID"].nunique():,} códigos · {df["SITEID"].nunique():,} sitios
  · países detectados: {", ".join(sorted(p for p in sit["pais"].unique() if p != "REVISAR"))}
- Clasificación automática: **{cobertura:.0%}** de los códigos · {catalogo["codigo_maestro"].notna().sum():,} con código maestro
- {len(pendientes):,} códigos en la lista de pendientes (ordenada por stock disponible)
- Archivo: `{EXCEL_PATH.name}` (SHA-256 `{file_hash[:16]}…`) · resultados en `{SALIDA}`

**Qué es automático y qué requiere intervención cuando llega un archivo nuevo**

| Paso | Automático | Requiere persona |
|---|---|---|
| Carga, perfil y conversión numérica | ✔ | solo si aparecen formatos numéricos nuevos |
| País y tipo de sitio | ✔ reglas + evidencia de códigos | sitios nuevos sin evidencia → `PAIS_MANUAL` |
| Nombre, unidades, marca | ✔ | nuevas variantes de marca → `ALIAS_MARCA` |
| Familia y categoría | ✔ cuando nombre o etiqueta coinciden | conflictos y "sin regla" (lista de pendientes) |
| Código maestro | ✔ registro persistente | — |
| Duplicados | ✔ detección | aprobar o rechazar cada grupo |

**Decisiones que necesita RLA**
1. Aprobar la taxonomía (13 categorías, 37 familias) y el formato de código `CAT-FAM-NNNNN`.
2. Validar país y tipo de los sitios en REVISAR; confirmar la moneda de cada país antes de sumar costos.
3. Qué hacer con los productos marcados "no usar / eliminar" y con los sitios de descarte y ajuste.
4. Revisar los grupos candidatos a duplicado (empezando por los que tienen stock).
'''))